# 00b POS Verification Workbooks

Task C of the protocol, redesigned for speed. Instead of assigning a tag to
every token from scratch, you judge the tag the model already produced. Roughly
3 seconds per token instead of 8.

**Note on which tool is being validated.** POS tags come from spaCy
`ru_core_news_sm`, not from razdel — razdel only tokenises and segments. This
notebook therefore works on spaCy's own tokens, restricted to alphabetic ones,
which is exactly the set the pipeline computes `pos_*_ratio` over
(`[t for t in doc if t.is_alpha]`).

## Why there are two files, and why the order matters

Verification is faster, but it is also biased: shown a plausible-looking wrong
tag, an annotator accepts it. The bias runs one way — it inflates accuracy —
and inflated accuracy is exactly what a reviewer questioning the tools will
attack.

So this notebook writes two workbooks:

| File | Tokens | What you see | Do it |
|---|---|---|---|
| `C_pos_blind.xlsx` | 100 | token and context only | **first** |
| `C_pos_verify.xlsx` | ≈ 320 | token, context, and the predicted tag | second |

The 100 blind tokens are a subset of the 320. Annotating them cold, before ever
seeing a prediction, gives an unbiased reading on the same items. Notebook `01`
compares the two and tells you which figure to report.

**Do not open `C_pos_verify.xlsx` until the blind file is finished.** Once you
have seen the predictions you cannot un-see them, and the blind check is spent.

The check is a sanity check, not a powered test: with 100 tokens and an error
rate around 5% you are looking at a handful of discordant items. Report it as
such — it detects gross acquiescence, not a two-point difference.


In [ ]:
!pip install razdel openpyxl spacy --quiet
!python -m spacy download ru_core_news_sm --quiet

## Parameters

`SEED` must match the one used in `00_draw_sample.ipynb`, so that the passages
here are the same ones already in the workbook.


In [ ]:
import os

BASE_PATH    = os.environ.get("KIDLIT_BASE", "./")
SOURCE_RUS_O = BASE_PATH + "kid_lit_100_ru.csv"
SOURCE_RUS_T = BASE_PATH + "kid_lit_100_foreign.csv"
MANIFEST     = "./sample_manifest.csv"

WORK_DIR = "./private_sheets"     # contains text — never commit

SEED       = 20260807
BLIND_N    = 100                  # tokens annotated cold, as a bias check
POS_SENTS  = 1                    # first N sentences of each passage

os.makedirs(WORK_DIR, exist_ok=True)

## Rebuild the sampled sentences

The manifest records which passage came from which text, so the sentences are
recovered deterministically rather than resampled.


In [ ]:
import pandas as pd
from razdel import sentenize

def load_texts(paths):
    frames = []
    for origin, path in zip(("RUS-O", "RUS-T"), paths):
        d = pd.read_csv(path, sep=";", encoding="utf-8-sig")[["id", "text"]].copy()
        d["text_id"] = [f"{origin}-{int(i):02d}" for i in d.id]
        frames.append(d.drop(columns="id"))
    return pd.concat(frames, ignore_index=True)

texts    = load_texts([SOURCE_RUS_O, SOURCE_RUS_T]).set_index("text_id").text
manifest = pd.read_csv(MANIFEST)

rows = []
for m in manifest.itertuples(index=False):
    sents = [s.text.strip() for s in sentenize(str(texts[m.text_id]))]
    sents = [s for s in sents if s]
    for k in range(POS_SENTS):
        rows.append(dict(passage_id=m.passage_id, text_id=m.text_id,
                         origin_type=m.origin_type, genre=m.genre,
                         sent_index=m.sent_index_from + k,
                         sentence=sents[m.sent_index_from + k]))

sampled = pd.DataFrame(rows)
print(f"{len(sampled)} sentences from {sampled.text_id.nunique()} texts")
sampled.head(3)

## Tag with the pipeline's own model

Same model, same alphabetic-token filter as the metric pipeline, so what you
verify is literally what the study measured.


In [ ]:
import spacy

nlp = spacy.load("ru_core_news_sm")

tokens = []
for s in sampled.itertuples(index=False):
    doc = nlp(s.sentence)
    for j, t in enumerate(t for t in doc if t.is_alpha):
        tokens.append(dict(passage_id=s.passage_id, text_id=s.text_id,
                           origin_type=s.origin_type, genre=s.genre,
                           sent_index=s.sent_index, token_index=j,
                           token=t.text, context=s.sentence, pred_pos=t.pos_))

tokens = pd.DataFrame(tokens)
print(f"{len(tokens)} alphabetic tokens")
print(tokens.pred_pos.value_counts().to_dict())

## Draw the blind subset

Stratified by predicted tag so the check covers the rarer categories, then
shuffled so the order carries no information about the model's output.


In [ ]:
import numpy as np

rng = np.random.default_rng(SEED)

# proportional allocation with at least one token per predicted tag
share = tokens.pred_pos.value_counts(normalize=True)
quota = (share * BLIND_N).round().astype(int).clip(lower=1)

picked = []
for tag, n in quota.items():
    pool = tokens.index[tokens.pred_pos == tag].to_numpy()
    picked.extend(rng.choice(pool, size=min(n, len(pool)), replace=False))

blind_idx = pd.Index(picked)[:BLIND_N]
blind = tokens.loc[blind_idx].sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"blind subset: {len(blind)} tokens")
print(blind.pred_pos.value_counts().to_dict())

## Write the workbooks

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

UD_TAGS = ["NOUN", "PROPN", "VERB", "AUX", "ADJ", "ADV", "PRON", "DET", "NUM",
           "ADP", "CCONJ", "SCONJ", "PART", "INTJ", "X"]
HDR = PatternFill("solid", fgColor="DDDDDD")


def write_sheet(path, title, frame, widths, wrap, validations, banner):
    wb = Workbook()
    ws = wb.active
    ws.title = title
    ws.append([banner])
    ws["A1"].font = Font(bold=True, color="9C0006")
    ws.append(list(frame.columns))
    for cell in ws[2]:
        cell.font, cell.fill = Font(bold=True), HDR
    for row in frame.itertuples(index=False):
        ws.append(["" if pd.isna(v) else v for v in row])
    ws.freeze_panes = "A3"
    cols = list(frame.columns)
    for col, w in widths.items():
        ws.column_dimensions[get_column_letter(cols.index(col) + 1)].width = w
    for col in wrap:
        i = cols.index(col) + 1
        for r in range(3, len(frame) + 3):
            ws.cell(r, i).alignment = Alignment(wrap_text=True, vertical="top")
    for col, options in validations:
        letter = get_column_letter(cols.index(col) + 1)
        dv = DataValidation(type="list", formula1='"' + ",".join(options) + '"',
                            allow_blank=True, showErrorMessage=True)
        ws.add_data_validation(dv)
        dv.add(f"{letter}3:{letter}{len(frame) + 2}")
    wb.save(path)
    return path


# ---- 1. blind: no predicted tag anywhere in the file ----------------------
b = blind[["passage_id", "text_id", "origin_type", "sent_index",
           "token_index", "token", "context"]].copy()
b["gold_pos"] = None
b["note"] = None

assert "pred_pos" not in b.columns, "the blind sheet must not carry predictions"

write_sheet(f"{WORK_DIR}/C_pos_blind.xlsx", "C_pos_blind", b,
            widths={"context": 70, "token": 18, "passage_id": 11,
                    "text_id": 11, "origin_type": 11, "note": 24},
            wrap=("context",),
            validations=[("gold_pos", UD_TAGS)],
            banner="DO THIS FIRST. Assign the tag yourself. "
                   "Do not open C_pos_verify.xlsx until this file is complete.")

# ---- 2. verification: the predicted tag is shown -------------------------
v = tokens[["passage_id", "text_id", "origin_type", "sent_index",
            "token_index", "token", "context", "pred_pos"]].copy()
v["tag_ok"] = None
v["correct_pos"] = None
v["is_onomatopoeia"] = None
v["is_diminutive"] = None
v["note"] = None

write_sheet(f"{WORK_DIR}/C_pos_verify.xlsx", "C_pos_verify", v,
            widths={"context": 70, "token": 18, "pred_pos": 11,
                    "correct_pos": 12, "passage_id": 11, "text_id": 11,
                    "origin_type": 11, "note": 24},
            wrap=("context",),
            validations=[("tag_ok", ["1", "0"]),
                         ("correct_pos", UD_TAGS),
                         ("is_onomatopoeia", ["1", "0"]),
                         ("is_diminutive", ["1", "0"])],
            banner="SECOND. Judge the tag in pred_pos. When tag_ok = 0, "
                   "put the right tag in correct_pos.")

print(f"{WORK_DIR}/C_pos_blind.xlsx    {len(b):>4} tokens   ~13 min")
print(f"{WORK_DIR}/C_pos_verify.xlsx   {len(v):>4} tokens   ~16 min")

## How to fill them in

**`C_pos_blind.xlsx` — first, ~13 minutes.** One column to fill: `gold_pos`.
Assign the UD tag yourself from the token and its context. There is no
prediction anywhere in this file, and there must not be: if you look one up,
the check is void.

**`C_pos_verify.xlsx` — second, ~16 minutes.**

| Column | Fill with |
|---|---|
| `tag_ok` | 1 if `pred_pos` is right, 0 if not |
| `correct_pos` | the right tag — only when `tag_ok` = 0 |
| `is_onomatopoeia`, `is_diminutive` | 1 for yes, blank for no |

Fill `correct_pos` even though it is extra work: errors are rare, so it costs a
couple of minutes, and without it there is no confusion matrix — and the
confusion matrix is what shows *where* the model fails on 0+ material, which is
the substance of the reviewer's objection.

The two hard-case flags matter most on the tokens the model gets wrong. If
onomatopoeia turns out to be tagged `NOUN` or `X`, that is a finding worth
reporting, not something to hide behind an aggregate.

Do not sort or filter the verification sheet by `pred_pos`. Reviewing a block
of identical predictions is faster, but it is also where rubber-stamping
happens.

When both files are done, export each to CSV into `private_sheets/` as
`C_pos_blind.csv` and `C_pos_verify.csv`, then run notebook `01`.
